In [1]:
import requests
import pandas as pd
import json
import os

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
#Set API Base URL
BASE_URL = "http://127.0.0.1:8000"

print("API Base URL:", BASE_URL)

API Base URL: http://127.0.0.1:8000


In [3]:
#Check API Health

response = requests.get(f"{BASE_URL}/")

print("Status Code:", response.status_code)
print(json.dumps(response.json(), indent=4))

Status Code: 200
{
    "message": "Student Memory Personalization API is running.",
    "component": "Component 3: Memory (Student Personalization)",
    "memory_layers": [
        "short-term memory",
        "long-term memory",
        "concept-based memory"
    ],
    "storage_modes": {
        "csv_memory": "Pre-generated memory tables from ASSISTments dataset",
        "sqlite_memory": "Dynamic memory updates from live interactions"
    },
    "available_endpoints": [
        "GET /memory/student/{student_id}",
        "GET /memory/context/{student_id}",
        "GET /memory/fapr-context/{student_id}",
        "GET /memory/meta-session/{student_id}/{session_id}",
        "GET /memory/student/{student_id}/concept/{concept_name}",
        "GET /memory/student/{student_id}/interactions",
        "POST /memory/update"
    ]
}


In [4]:
#Define Test Student and Session
student_id = 999892
session_id = 7001

print("Test Student ID:", student_id)
print("Test Session ID:", session_id)

Test Student ID: 999892
Test Session ID: 7001


In [5]:
#Create Test Interactions

test_interactions = [
    {
        "student_id": student_id,
        "session_id": session_id,
        "problem_id": 301,
        "concept_name": "Percent Of",
        "correct": 0,
        "attempt_count": 2,
        "hint_count": 1,
        "hint_total": 3,
        "response_time_ms": 45000
    },
    {
        "student_id": student_id,
        "session_id": session_id,
        "problem_id": 302,
        "concept_name": "Percent Of",
        "correct": 1,
        "attempt_count": 1,
        "hint_count": 0,
        "hint_total": 3,
        "response_time_ms": 30000
    },
    {
        "student_id": student_id,
        "session_id": session_id,
        "problem_id": 303,
        "concept_name": "Circle Graph",
        "correct": 1,
        "attempt_count": 1,
        "hint_count": 0,
        "hint_total": 2,
        "response_time_ms": 28000
    }
]

test_interactions

[{'student_id': 999892,
  'session_id': 7001,
  'problem_id': 301,
  'concept_name': 'Percent Of',
  'correct': 0,
  'attempt_count': 2,
  'hint_count': 1,
  'hint_total': 3,
  'response_time_ms': 45000},
 {'student_id': 999892,
  'session_id': 7001,
  'problem_id': 302,
  'concept_name': 'Percent Of',
  'correct': 1,
  'attempt_count': 1,
  'hint_count': 0,
  'hint_total': 3,
  'response_time_ms': 30000},
 {'student_id': 999892,
  'session_id': 7001,
  'problem_id': 303,
  'concept_name': 'Circle Graph',
  'correct': 1,
  'attempt_count': 1,
  'hint_count': 0,
  'hint_total': 2,
  'response_time_ms': 28000}]

In [8]:
#Submit Test Interactions to Memory API

update_results = []

for interaction in test_interactions:
    response = requests.post(
        f"{BASE_URL}/memory/update",
        json=interaction
    )
    
    result = response.json()
    update_results.append(result)
    
    print("Status Code:", response.status_code)
    print("Student:", interaction["student_id"])
    print("Concept:", interaction["concept_name"])
    print("Correct:", interaction["correct"])
    print("-" * 50)

Status Code: 200
Student: 999892
Concept: Percent Of
Correct: 0
--------------------------------------------------
Status Code: 200
Student: 999892
Concept: Percent Of
Correct: 1
--------------------------------------------------
Status Code: 200
Student: 999892
Concept: Circle Graph
Correct: 1
--------------------------------------------------


In [10]:
#Call FAPR-LB Context Endpoint

fapr_response = requests.get(
    f"{BASE_URL}/memory/fapr-context/{student_id}",
    params={
        "session_id": session_id,
        "current_skill_id": "Percent Of",
        "limit": 5
    }
)

print("Status Code:", fapr_response.status_code)

fapr_context = fapr_response.json()
print(json.dumps(fapr_context, indent=4))

Status Code: 200
{
    "found": true,
    "source": "sqlite",
    "student_id": "999892",
    "session_id": "7001",
    "current_skill_id": "Percent Of",
    "recent_interactions": [
        {
            "skill_id": "Percent Of",
            "correct": 0,
            "hint_count": 1,
            "attempt_count": 2,
            "response_time_ms": 45000.0
        },
        {
            "skill_id": "Percent Of",
            "correct": 1,
            "hint_count": 0,
            "attempt_count": 1,
            "response_time_ms": 30000.0
        },
        {
            "skill_id": "Circle Graph",
            "correct": 1,
            "hint_count": 0,
            "attempt_count": 1,
            "response_time_ms": 28000.0
        }
    ],
    "current_attempt": {
        "skill_id": "Circle Graph",
        "correct": 1,
        "hint_count": 0,
        "attempt_count": 1,
        "response_time_ms": 28000.0
    },
    "previous_repair": {
        "repair_action": null,
        "outcome

In [11]:
#Validate FAPR-LB Schema

required_fapr_fields = [
    "student_id",
    "session_id",
    "current_skill_id",
    "recent_interactions",
    "current_attempt",
    "previous_repair",
    "last_student_utterance",
    "last_tutor_response"
]

fapr_schema_results = []

for field in required_fapr_fields:
    fapr_schema_results.append({
        "field": field,
        "available": field in fapr_context
    })

fapr_schema_df = pd.DataFrame(fapr_schema_results)

fapr_schema_df

,field,available
0,student_id,True
1,session_id,True
2,current_skill_id,True
3,recent_interactions,True
4,current_attempt,True
5,previous_repair,True
6,last_student_utterance,True
7,last_tutor_response,True


In [12]:
#Validate Recent Interaction Fields

required_recent_interaction_fields = [
    "skill_id",
    "correct",
    "hint_count",
    "attempt_count",
    "response_time_ms"
]

recent_interactions = fapr_context.get("recent_interactions", [])

interaction_field_results = []

for index, interaction in enumerate(recent_interactions):
    for field in required_recent_interaction_fields:
        interaction_field_results.append({
            "interaction_index": index,
            "field": field,
            "available": field in interaction
        })

interaction_field_df = pd.DataFrame(interaction_field_results)

interaction_field_df

,interaction_index,field,available
0,0,skill_id,True
1,0,correct,True
2,0,hint_count,True
3,0,attempt_count,True
4,0,response_time_ms,True
5,1,skill_id,True
6,1,correct,True
7,1,hint_count,True
8,1,attempt_count,True
9,1,response_time_ms,True


In [13]:
# FAPR-LB Compatibility Test

fapr_test_status = "PASS"

if not fapr_context.get("found", False):
    fapr_test_status = "FAIL"

for field in required_fapr_fields:
    if field not in fapr_context:
        fapr_test_status = "FAIL"

for interaction in recent_interactions:
    for field in required_recent_interaction_fields:
        if field not in interaction:
            fapr_test_status = "FAIL"

fapr_test_result = {
    "test_name": "FAPR-LB context schema compatibility",
    "expected_component": "FAPR-LB",
    "status": fapr_test_status,
    "recent_interaction_count": len(recent_interactions)
}

fapr_test_result

{'test_name': 'FAPR-LB context schema compatibility',
 'expected_component': 'FAPR-LB',
 'status': 'PASS',
 'recent_interaction_count': 3}

In [14]:
#Call Meta-Agent Session Endpoint

meta_response = requests.get(
    f"{BASE_URL}/memory/meta-session/{student_id}/{session_id}"
)

print("Status Code:", meta_response.status_code)

meta_context = meta_response.json()
print(json.dumps(meta_context, indent=4))

Status Code: 200
{
    "found": true,
    "source": "sqlite",
    "student_id": "999892",
    "session_id": "7001",
    "attempt_count": 3,
    "attempts": [
        {
            "skill": "Percent Of",
            "correct": 0
        },
        {
            "skill": "Percent Of",
            "correct": 1
        },
        {
            "skill": "Circle Graph",
            "correct": 1
        }
    ],
    "misconceptions": [],
    "integration_note": {
        "target_component": "Meta-Agent",
        "purpose": "Chronological session attempts for BKT mastery tracking, knowledge graph updates, and learning path generation.",
        "important_constraints": [
            "Skill names should match the canonical ASSISTments skill list.",
            "correct must be binary: 0 or 1.",
            "Attempts are returned in chronological order.",
            "Misconceptions are optional and currently returned as an empty list."
        ]
    }
}


In [15]:
#Validate Meta-Agent Schema

required_meta_fields = [
    "student_id",
    "session_id",
    "attempts",
    "misconceptions"
]

meta_schema_results = []

for field in required_meta_fields:
    meta_schema_results.append({
        "field": field,
        "available": field in meta_context
    })

meta_schema_df = pd.DataFrame(meta_schema_results)

meta_schema_df

,field,available
0,student_id,True
1,session_id,True
2,attempts,True
3,misconceptions,True


In [16]:
#Validate Attempt Fields

required_attempt_fields = [
    "skill",
    "correct"
]

attempts = meta_context.get("attempts", [])

attempt_field_results = []

for index, attempt in enumerate(attempts):
    for field in required_attempt_fields:
        attempt_field_results.append({
            "attempt_index": index,
            "field": field,
            "available": field in attempt
        })

attempt_field_df = pd.DataFrame(attempt_field_results)

attempt_field_df

,attempt_index,field,available
0,0,skill,True
1,0,correct,True
2,1,skill,True
3,1,correct,True
4,2,skill,True
5,2,correct,True


In [18]:
#Validate Binary Correctness

binary_correctness_results = []

for index, attempt in enumerate(attempts):
    correct_value = attempt.get("correct")
    binary_correctness_results.append({
        "attempt_index": index,
        "correct_value": correct_value,
        "is_binary": correct_value in [0, 1]
    })

binary_correctness_df = pd.DataFrame(binary_correctness_results)

binary_correctness_df

,attempt_index,correct_value,is_binary
0,0,0,True
1,1,1,True
2,2,1,True


In [19]:
#Validate Chronological Order

expected_attempt_sequence = [
    {"skill": "Percent Of", "correct": 0},
    {"skill": "Percent Of", "correct": 1},
    {"skill": "Circle Graph", "correct": 1}
]

actual_attempt_sequence = [
    {
        "skill": attempt["skill"],
        "correct": attempt["correct"]
    }
    for attempt in attempts
]

chronological_order_correct = actual_attempt_sequence == expected_attempt_sequence

chronological_order_result = {
    "test_name": "Meta-Agent chronological order validation",
    "expected_sequence": expected_attempt_sequence,
    "actual_sequence": actual_attempt_sequence,
    "status": "PASS" if chronological_order_correct else "FAIL"
}

chronological_order_result


{'test_name': 'Meta-Agent chronological order validation',
 'expected_sequence': [{'skill': 'Percent Of', 'correct': 0},
  {'skill': 'Percent Of', 'correct': 1},
  {'skill': 'Circle Graph', 'correct': 1}],
 'actual_sequence': [{'skill': 'Percent Of', 'correct': 0},
  {'skill': 'Percent Of', 'correct': 1},
  {'skill': 'Circle Graph', 'correct': 1}],
 'status': 'PASS'}

In [20]:
#Meta-Agent Compatibility Test

meta_test_status = "PASS"

if not meta_context.get("found", False):
    meta_test_status = "FAIL"

for field in required_meta_fields:
    if field not in meta_context:
        meta_test_status = "FAIL"

for attempt in attempts:
    for field in required_attempt_fields:
        if field not in attempt:
            meta_test_status = "FAIL"
    if attempt.get("correct") not in [0, 1]:
        meta_test_status = "FAIL"

if not chronological_order_correct:
    meta_test_status = "FAIL"

meta_test_result = {
    "test_name": "Meta-Agent session export compatibility",
    "expected_component": "Meta-Agent",
    "status": meta_test_status,
    "attempt_count": len(attempts)
}

meta_test_result

{'test_name': 'Meta-Agent session export compatibility',
 'expected_component': 'Meta-Agent',
 'status': 'PASS',
 'attempt_count': 3}

In [21]:
#Create Final Test Results Table

integration_test_results = pd.DataFrame([
    fapr_test_result,
    meta_test_result,
    chronological_order_result
])

integration_test_results

,test_name,expected_component,status,recent_interaction_count,attempt_count,expected_sequence,actual_sequence
0,FAPR-LB context schema compatibility,FAPR-LB,PASS,3.0,NaN,NaN,NaN
1,Meta-Agent session export compatibility,Meta-Agent,PASS,NaN,3.0,NaN,NaN
2,Meta-Agent chronological order validation,NaN,PASS,NaN,NaN,"[{'skill': 'Percent Of', 'correct': 0}, {'skil...","[{'skill': 'Percent Of', 'correct': 0}, {'skil..."


In [27]:
import os
import json

# Create output folders
os.makedirs("../outputs/tables", exist_ok=True)
os.makedirs("../outputs/api_results", exist_ok=True)

# Create summary if not already created
summary = integration_test_results["status"].value_counts().reset_index()
summary.columns = ["status", "count"]

# Save test results
integration_test_results.to_csv(
    "../outputs/tables/integration_context_test_results.csv",
    index=False
)

# Save summary
summary.to_csv(
    "../outputs/tables/integration_context_test_summary.csv",
    index=False
)

# Save FAPR-LB context sample
with open("../outputs/api_results/fapr_context_sample.json", "w") as f:
    json.dump(fapr_context, f, indent=4)

# Save Meta-Agent context sample
with open("../outputs/api_results/meta_session_sample.json", "w") as f:
    json.dump(meta_context, f, indent=4)

print("Integration context test outputs saved successfully.")
display(summary)

Integration context test outputs saved successfully.


,status,count
0,PASS,3


In [29]:
#Save Outputs

os.makedirs("../outputs/tables", exist_ok=True)
os.makedirs("../outputs/api_results", exist_ok=True)

integration_test_results.to_csv(
    "../outputs/tables/integration_context_test_results.csv",
    index=False
)

summary.to_csv(
    "../outputs/tables/integration_context_test_summary.csv",
    index=False
)

with open("../outputs/api_results/fapr_context_sample.json", "w") as f:
    json.dump(fapr_context, f, indent=4)

with open("../outputs/api_results/meta_session_sample.json", "w") as f:
    json.dump(meta_context, f, indent=4)

print("Integration context test outputs saved successfully.")

Integration context test outputs saved successfully.
